# **Activity - FFN/RNN/LSTM for Language Modeling**

**Student:** Louis G. Binwag  
**References:** Course notebooks (`.ipynb`)  
**Instructions:** Develop a Filipino/Tagalog language model using pre-Transformer architectures (FFN, RNN, LSTM, etc.).

1. Find a large enough Filipino/Tagalog corpus.
2. Create a WordLSTM class for your LM.
3. This model must be be deeper >3 hidden layers. 
4. Create a different LM classwith a different pre-Transformer architecture (FNN, RNN, etc.) that is not LSTM-based. 
5. Train the two LMS with the Filipino/Tagalog corpus.
6. Provide 3 different Filipino/Tagalog prompts.
7. Induce the models to generate over these prompts, and compare the outputs.

**Expected Output:** Jupyter Notebook

In [13]:
import pandas as pd

In [14]:
import pandas as pd

file_path = "tgl_sentences.tsv.bz2"

df = pd.read_csv(file_path,
                compression='bz2',
                sep='\t',
                header=None,   
                names=["Index", "Lang", "Sentence"],  
                low_memory=False)

# Check first few rows
print(df.head())

# Save to CSV
df.to_csv("tagalog-sentences.csv", index=False, encoding="utf-8")

# Save the "Sentence" column to a plain text file
df["Sentence"].to_csv("tagalog-sentences.txt", index=False, header=False, encoding="utf-8")

    Index Lang                                    Sentence
0  433280  tgl                               Gutom na ako.
1  508360  tgl  Paano mo sabihin ang "good bye" sa Aleman?
2  508363  tgl                 Diyos ang lumikha sa mundo.
3  508366  tgl                   Pwede ba kitang tulungan?
4  508369  tgl                          Paano ang opisina?


In [15]:
# Preprocess
import pandas as pd
import re

# Load CSV
df = pd.read_csv("tagalog-sentences.csv")

# Keep only the Sentence column
df = df[["Sentence"]].dropna()

# Lowercase and basic cleaning
def clean_text(text):
    text = text.lower()                           # lowercase
    text = re.sub(r"[^a-zA-ZÀ-ž\s]", "", text)    # remove numbers, punctuation but keep letters
    text = re.sub(r"\s+", " ", text).strip()      # remove extra spaces
    return text

df["Sentence"] = df["Sentence"].apply(clean_text)

# Save cleaned dataset
df.to_csv("tagalog-sentences-clean.csv", index=False, encoding="utf-8")

print(df.head(10))

                                            Sentence
0                                       gutom na ako
1            paano mo sabihin ang good bye sa aleman
2                         diyos ang lumikha sa mundo
3                           pwede ba kitang tulungan
4                                  paano ang opisina
5                          magdadala ako sa iyo doon
6                                    kumusta po kayo
7  ang lahat ng taoy isinilang na malaya at panta...
8  tigilan mo na ang asawa ko kasi nakakahiya sa ...
9      pakiusap intindihin mo na lang ang trabaho mo


In [16]:
import torch
import torch.nn as nn
import torch.optim as optim
import requests
import os
import re
from collections import Counter
from tqdm import tqdm

In [17]:
def get_best_device():
    if torch.cuda.is_available():
        return torch.device('cuda')
    elif torch.backends.mps.is_available():
        return torch.device('mps')
    else:
        return torch.device('cpu')


def download_dataset(url, data_path):
    if not os.path.exists(data_path):
        with open(data_path, 'w') as f:
            f.write(requests.get(url).text)
    with open(data_path, 'r') as f:
        text = f.read()
    return text


def tokenize(text):
    text = re.sub(r'[\W_]+', ' ', text.lower())
    return text.split()

def build_vocab(words):
    vocab = sorted(set(words))
    word2int = {w: i for i, w in enumerate(vocab)}
    int2word = {i: w for i, w in enumerate(vocab)}
    return vocab, word2int, int2word

def encode(s, word2int):
    return [word2int[w] for w in tokenize(s)]

def decode(l, int2word):
    return ' '.join([int2word[i] for i in l])

def get_batch(data, batch_size, block_size, device):
    ix = torch.randint(len(data) - block_size - 1, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    return x.to(device), y.to(device)

In [18]:
class WordLSTM(nn.Module):
    """
    Multi-layer LSTM model for word-level language modeling.
    Args:
        vocab_size (int): Size of the vocabulary.
        hidden_size (int): Number of hidden units in each LSTM layer.
        num_layers (int): Number of LSTM layers.
        word2int (dict): Word to index mapping (for reference).
        int2word (dict): Index to word mapping (for reference).
    """
    
    def __init__(self, vocab_size, hidden_size, num_layers, word2int, int2word):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, hidden_size)
        self.lstm = nn.LSTM(hidden_size, hidden_size, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, vocab_size)
        self.word2int = word2int
        self.int2word = int2word

    def forward(self, x, hidden=None):
        """
        Forward pass through the model.
        Args:
            x (Tensor): Input tensor of word indices.
            hidden (tuple, optional): Hidden state for LSTM.
        Returns:
            tuple: (output logits, new hidden state)
        """
        x = self.embed(x)
        out, hidden = self.lstm(x, hidden)
        out = self.fc(out)
        return out, hidden

def train_model(model, data, vocab_size, device, epochs=2, batch_size=64, block_size=30, lr=2e-3, steps_per_epoch=200):
    """
    Train the LSTM model on the dataset.
    Args:
        model (nn.Module): The LSTM model.
        data (Tensor): The dataset as a tensor of word indices.
        vocab_size (int): Size of the vocabulary.
        device (torch.device): Device to train on.
        epochs (int): Number of epochs.
        batch_size (int): Batch size.
        block_size (int): Sequence length.
        lr (float): Learning rate.
        steps_per_epoch (int): Number of steps per epoch.
    Returns:
        nn.Module: The trained model.
    """
    optimizer = optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.CrossEntropyLoss()
    for epoch in range(epochs):
        model.train()
        total_loss = 0
        pbar = tqdm(range(steps_per_epoch), desc=f"Epoch {epoch+1}")
        for step in pbar:
            # Get a batch of input word indices(x) and 
            # target word indices (y) sequences
            x, y = get_batch(data, batch_size, block_size, device)

            optimizer.zero_grad()
            # Forward pass: logits are the raw, unnormalized scores for
            # each word in the vocab
            # logits: (batch_size, block_size, vocab_size)
            logits, _ = model(x)

            # Reshape logits and targets for loss computation
            logits_flat = logits.view(-1, vocab_size)  # (batch_size*block_size, vocab_size)
            y_flat = y.view(-1)  # (batch_size*block_size)

            # Compute cross-entropy loss between predicted logits and true targets
            # loss: average negative log likelihood over the batch
            loss = loss_fn(logits_flat, y_flat)

            # Backpropagation and optimization step
            loss.backward()
            optimizer.step()

            total_loss += loss.item()
            if (step+1) % 50 == 0:
                pbar.set_postfix({'loss': loss.item()})

        print(f"Epoch {epoch+1} avg loss: {total_loss/steps_per_epoch:.4f}")
    return model

def train_lstm(device=None, block_size=32, batch_size=64, hidden_size=512, num_layers=6, epochs=3, lr=2e-3, steps_per_epoch=200): # Changed to 6 layers 
    """
    Prepare data, build the model, and train the LSTM.
    Args:
        device (torch.device, optional): Device to use. If None, auto-selects best device.
    Returns:
        nn.Module: The trained LSTM model.
    """

    # Dataset
    # url = 'https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt'
    # data_path = 'tagalog-sentences.csv'
    # text = pd.read_csv("tagalog-sentences.csv")
    # words = tokenize(text)
    
    
    with open("tagalog-sentences.txt", "r", encoding="utf-8") as f:
        text = f.read()  # now text is a string

    # Now tokenize
    words = tokenize(text)

    
    
    vocab, word2int, int2word = build_vocab(words)
    vocab_size = len(vocab)
    data = torch.tensor([word2int[w] for w in words], dtype=torch.long)

    # Device selection (CUDA > MPS > CPU)
    if device is None:
        device = get_best_device()
    print(f"Using device: {device}")

    # Model initialization
    model = WordLSTM(vocab_size, hidden_size, num_layers, word2int, int2word).to(device)

    # Train the model
    trained_model = train_model(model, data, vocab_size, device, epochs, batch_size, block_size, lr, steps_per_epoch)
    return trained_model

def generate_text(model, prompt, device=None, max_tokens=64):
    """
    Generate text from a trained model given a prompt.
    Args:
        model (nn.Module): The trained LSTM model.
        prompt (str): The initial text prompt.
        device (torch.device, optional): Device to use. If None, auto-selects best device.
        length (int): Number of words to generate.
    Returns:
        None. Prints the generated text.
    """
    # Device selection (CUDA > MPS > CPU)
    if device is None:
        device = get_best_device()
    print(f"Using device: {device}")

    model.eval()

    # Encode the prompt into word indices
    # input_ids: list of indices for the prompt words
    input_ids = encode(prompt, model.word2int)
    
    # Convert to tensor and add batch dimension
    # input_seq: shape (1, prompt_length)
    input_seq = torch.tensor(input_ids, dtype=torch.long).unsqueeze(0).to(device)
    
    hidden = None   # Initial hidden state (None = zeros)
    generated = list(input_ids)  # Start with the prompt

    for _ in range(max_tokens):
        # Forward pass: get logits for the next word
        # logits: (1, seq_len, vocab_size), only last position is relevant
        logits, hidden = model(input_seq, hidden)
        
        # Get probabilities for the next word (softmax over vocab)
        probs = torch.softmax(logits[:, -1, :], dim=-1)  # (1, vocab_size)

        # Sample the next word index from the probability distribution
        next_id = torch.multinomial(probs, num_samples=1).item()
        generated.append(next_id)

        # Prepare input for next step: the newly generated word
        input_seq = torch.tensor([[next_id]], dtype=torch.long).to(device)

    # Decode the generated indices back to words and print
    print("\nGenerated text:\n", decode(generated, model.int2word))


In [19]:
# Train the LSTM model
model = train_lstm()

Using device: cpu


Epoch 1: 100%|██████████| 200/200 [04:15<00:00,  1.28s/it, loss=6.82]


Epoch 1 avg loss: 7.0368


Epoch 2: 100%|██████████| 200/200 [05:30<00:00,  1.65s/it, loss=6.87]


Epoch 2 avg loss: 6.8784


Epoch 3: 100%|██████████| 200/200 [05:15<00:00,  1.58s/it, loss=6.76]

Epoch 3 avg loss: 6.7589


In [20]:
generate_text(model, prompt="Bakit ba kaya,", max_tokens=32)

Using device: cpu

Generated text:
 bakit ba kaya pong handang araw ang bakit huwag pinayagan sa gawin kaya selpon na tom si barista puno umiinom lang tom nasa naman si manggagawa ko sa na ng walang ito tibok na kasangkot


In [21]:
generate_text(model, prompt="Kundi man tayo sa, ", max_tokens=32)

Using device: cpu

Generated text:
 kundi man tayo sa kailangan oras tumataas sa ng mong akin po pa ng pinapakain nating kanilang kay sa ay ang wikang lahi kuripot mga na mga ang wala bang tulugan ito ang makita gusto nakalimutan


In [22]:
generate_text(model, prompt="Ayon sa sabi ni Jose Rizal", max_tokens=32)

Using device: cpu

Generated text:
 ayon sa sabi ni jose rizal asukal nagtataka ni wally ako kung isip ang ba malaking isang letrang ilang napunta magulang siya nang huwag ang sa mayroon maging kinakailangang ko ng nakita pagdaing naroon ako ibababa tradisyon masuwerte


### **Short Conclusion**

Followed the same process as we did in the lecture, and was able to change the input dataset into the `Tagalog Dataset` I got through [**Tatoeba**](https://tatoeba.org/en/downloads)

The three outputs seem to be a little `scuffed` and probably does not make much sense. This could be because the dataset may not be trained enough on a rich enough dataset when it comes to Tagalog.

Although the third generated text seem a little more complete in thought, it is still missing concrete flow of thought which may be hard without a `richer training data` and `multiple training stages`.

Whereas all of them tend to plug in various Tagalog words that seem correct with the previous word, but not necessarily creating a full sentence. 

I also think that due to the fact that many Tagalog words have minor spelling changes like:
- "Bakit" & "Baket" (Why)
- "Hindi" & "Indi" (No)
- "Bituwin" & "Bituin" (Star)
- "Marsmalo" & "Marshmalo" (Marshmallow)

Which have very minor spelling changes, but I think this also affects the model's training process considering it is case-sensitive and cannot distinguish typographical errors of words (Hence "Bakit" and "Baket" are considered two whole different tokens when they *should* mean the same)

### **Some things I want to know though**
- What does the % lost mean during training period?
- How can I 'further' clean my dataset considering the duplicated or very similar words with minor spelling changes? (Especially on a dataset with over 100k rows?)
- How can I make my system use the GPU..? Cuda should be available since I'm running on Nvidia as well